<a href="https://colab.research.google.com/github/Selvapriya05/Selvapriya-Codeboosters-2026/blob/main/Day_7/Day_7_Miniproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# #Mini project step 1 - Setup chromadb collection

# client = chromadb.EphemeraClient()

# notes_collection = client.get_or_create_collection(
#     name="college_notes"
# )

# print("ChromaDB collection 'college_notes' created")
# print(f"Documents in collection: {notes_collection.count()}")

In [2]:
# #Mini project step 3 - Run semantic Queries

# query_1 = "How do I fix messy and incomplete data?"

# results_1 = notes_collection.query(
#     query_texts=[query_1],
#     n_results=3
# )

# show_results(query_1, results_1)

In [5]:
!pip install chromadb
!pip install sentence-transformers
!pip install pandas

In [6]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

In [7]:
df = pd.read_csv("college_notes.csv")

df.head()

,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
client = chromadb.Client()

collection = client.create_collection(
    name="college_notes"
)

In [11]:
print(df.columns)

Index(['note_id', 'subject', 'topic', 'content'], dtype='object')


In [12]:
embeddings = model.encode(
    df["content"].tolist()
).tolist()

In [13]:
collection.add(
    ids=df["note_id"].astype(str).tolist(),
    documents=df["content"].tolist(),
    metadatas=[
        {
            "subject": subject,
            "topic": topic
        }
        for subject, topic in zip(df["subject"], df["topic"])
    ],
    embeddings=embeddings
)

In [14]:
collection.count()

15

In [15]:
def semantic_search(query):

    query_embedding = model.encode(
        [query]
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )

    print(f"\nQuery: {query}")
    print("-" * 50)

    for doc, distance in zip(
        results["documents"][0],
        results["distances"][0]
    ):

        print("Note:", doc)
        print("Distance:", round(distance,4))
        print()

In [16]:
semantic_search(
    "How can I predict house prices?"
)


Query: How can I predict house prices?
--------------------------------------------------
Note: A decision tree is a machine learning model that makes predictions by asking a series of yes or no questions about the features. It splits data at each node based on the feature that best separates the classes or reduces error.
Distance: 1.6826

Note: Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R-squared for regression. A good model generalizes well to new data it has not seen before.
Distance: 1.6893

Note: Random Forest is an ensemble learning method that builds multiple decision trees and combines their predictions. It is more accurate and robust than a single decision tree and handles overfitting well by averaging results from many trees.
Distance: 1.6954



In [17]:
semantic_search(
    "Big data processing tools"
)


Query: Big data processing tools
--------------------------------------------------
Note: Big Data refers to extremely large datasets that cannot be processed by traditional tools. Apache Spark is a distributed computing framework that processes big data across multiple machines simultaneously. PySpark is the Python API for Spark.
Distance: 0.7946

Note: Pandas is a Python library used for data manipulation and analysis. It provides the DataFrame data structure which is like a table with rows and columns. Common operations include reading CSV files filtering rows grouping data and creating new columns.
Distance: 1.3555

Note: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts line plots histograms and pie charts that help humans understand patterns in data.
Distance: 1.3758



In [18]:
semantic_search(
    "Protect sensitive information"
)


Query: Protect sensitive information
--------------------------------------------------
Note: A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
Distance: 1.7819

Note: Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizing formats.
Distance: 1.7917

Note: Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.
Distance: 1.7922



In [19]:
semantic_search(
    "Retrieve records from database"
)


Query: Retrieve records from database
--------------------------------------------------
Note: A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
Distance: 1.0799

Note: Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.
Distance: 1.5991

Note: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
Distance: 1.6133



In [20]:
semantic_search(
    "Real time data streaming"
)


Query: Real time data streaming
--------------------------------------------------
Note: Big Data refers to extremely large datasets that cannot be processed by traditional tools. Apache Spark is a distributed computing framework that processes big data across multiple machines simultaneously. PySpark is the Python API for Spark.
Distance: 1.3076

Note: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplotlib and Seaborn are used to create bar charts line plots histograms and pie charts that help humans understand patterns in data.
Distance: 1.4697

Note: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
Distance: 1.5451



In [21]:
results = collection.get(
    where={"subject":"Machine Learning"}
)

print(results["documents"])

['Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.', 'Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R-squared for regression. A good model generalizes well to new data it has not seen before.', 'Feature engineering is the process of selecting transforming and creating input variables that help machine learning models learn better. This includes encoding categorical variables scaling numerical features and handling missing data.', 'A decision tree is a machine learning model that makes predictions by asking a series of yes or no questions about the features. It splits data at each node based on the feature that best separates the classes or reduces error.', 'Random Forest is an ense

In [22]:
semantic_search(
    "predict values"
)


Query: predict values
--------------------------------------------------
Note: Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.
Distance: 1.4552

Note: Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R-squared for regression. A good model generalizes well to new data it has not seen before.
Distance: 1.5067

Note: A decision tree is a machine learning model that makes predictions by asking a series of yes or no questions about the features. It splits data at each node based on the feature that best separates the classes or reduces error.
Distance: 1.5313



In [23]:
query_embedding = model.encode(
    ["predict values"]
).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    where={"subject":"Machine Learning"}
)

print(results["documents"][0])

['Supervised learning is a type of machine learning where the model learns from labeled data. The model is given input features and correct output labels and it learns to predict outputs for new unseen inputs. Examples include classification and regression.', 'Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R-squared for regression. A good model generalizes well to new data it has not seen before.', 'A decision tree is a machine learning model that makes predictions by asking a series of yes or no questions about the features. It splits data at each node based on the feature that best separates the classes or reduces error.']
